In [ ]:
from bs4 import BeautifulSoup
import pandas as pd
import requests
import requests
import os
import time
from datetime import datetime
from utils import *


In [ ]:
# User-Agent atualizado
ARQUIVO = "decks_fabtcg.csv"
url = 'https://fabtcg.com/en/decklists/?query='

session = requests.Session()
session.headers.update(headers)

In [3]:
EVENT_MAPPING = {
    "tc": "tc",
    "upf": "calling",
    "calling": "calling",
    "battle hardened": "battle hardened",
    "proquest": "pro quest",
    "pro quest": "pro quest",
    "road to nationals": "road to nationals",
    "sunday showdown": "sunday showdown",
    "skirmish": "skirmish",
    "national": "national",
    "silver age spotlight": "silver age spotlight",
    "world championship": "world championships",
    "pro tour": "pro tour",
    "dumpster": "dumpster dive",
    "dev download": "dev download",
    "10k": "Road to $10k",
    "zero to eighty": "zero to eighty",
    "celebrational": "celebrational",
    "store champions": "store champions",
    "smash palace": "smash palace",
    "cat footprints shilin armed cup": "cat footprints shilin armed cup",
    "the savage lands showdown": "the savage lands showdown",
    "commoner": "commoner",
    "hong kong regional championship": "hong kong regional championship",
    "bulk up": "bulk up",
    "primed to fight": "primed to fight",
    "masterclass": "masterclass",
    "world tour": "world tour"
}

In [4]:
def classificar_evento(nome_evento):
    nome_lower = str(nome_evento).lower()
    for chave, valor in EVENT_MAPPING.items():
        if chave in nome_lower:
            return valor
    return "outro"

def extrair_dados():
    dados = []
    page = 1

    while True:
        url = f'https://fabtcg.com/decklists/page/{page}/'
        response = session.get(url)
        if response.status_code == 404:
            print(f"<============= Fim das páginas encontrado na página {page} =============>")
        if response.status_code != 200:
            print(f"Erro HTTP {response.status_code} na página {page}. Interrompendo.")
            break
            
        print(f"<============= Escaneando a página {page} =============>")
        soup = soup = BeautifulSoup(response.content, 'html.parser')
        table = soup.find('table')
        
        if table:
            linhas = table.find_all('tr')[1:]

            for linha in linhas:
                colunas = linha.find_all('td')

                dado = [coluna.text.strip() for coluna in colunas]

                coluna_deck = colunas[2] 
                elemento_a = coluna_deck.find('a')
                if elemento_a:
                    link_do_deck = elemento_a.get('href')
                    dado.append(link_do_deck)
                else:
                    dado.append("Sem link")

                if len(dado[3]) < 1 : continue

                tipo_evento = classificar_evento(dado[3])
                dado.append(tipo_evento)
                dados.append(dado)
        else:
            print(f"Erro, nehuma table encontrada na pagina: {page}")

        page +=1
        time.sleep(0.5)


    return dados

def arquivo_e_atual(caminho_arquivo):

    if not os.path.exists(caminho_arquivo):
        return False

    timestamp_arquivo = os.path.getmtime(caminho_arquivo)
    
    data_arquivo = datetime.fromtimestamp(timestamp_arquivo).date()
    
    data_hoje = datetime.now().date()

    return data_arquivo == data_hoje    


In [5]:

# Usamos a função que criamos para tomar a decisão
if arquivo_e_atual(ARQUIVO):
    print("O arquivo esta atualizado.")
    
else:
    print("O arquivo é antigo ou não existe. Atualizando.")
    
    if os.path.exists(ARQUIVO):
        print("Apagando o arquivo antigo")
        os.remove(ARQUIVO)
        print("Arquivo antigo apagado.")

    print("Iniciando a extração.")
    
    dados = extrair_dados()

    colunas_df = ["Pais", "Data", "Decklist", "Evento", "Formato", "Heroi", "Resultado","Link do deck", "Tipo de evento"]
    df = pd.DataFrame(dados, columns=colunas_df)

    df['Resultado'] = df['Resultado'].replace('', 'nao informado')

    df.to_csv(ARQUIVO, index=False)
    print("Dados extraidos e salvos localmente!")


df.head()

O arquivo é antigo ou não existe. Atualizando.
Iniciando a extração.
<============= Escaneando a página 1 =============>
<============= Escaneando a página 2 =============>
<============= Escaneando a página 3 =============>
<============= Escaneando a página 4 =============>
<============= Escaneando a página 5 =============>
<============= Escaneando a página 6 =============>
<============= Escaneando a página 7 =============>
<============= Escaneando a página 8 =============>
<============= Escaneando a página 9 =============>
<============= Escaneando a página 10 =============>
<============= Escaneando a página 11 =============>
<============= Escaneando a página 12 =============>
<============= Escaneando a página 13 =============>
<============= Escaneando a página 14 =============>
<============= Escaneando a página 15 =============>
<============= Escaneando a página 16 =============>
<============= Escaneando a página 17 =============>
<============= Escaneando a página 18 =

,Pais,Data,Decklist,Evento,Formato,Heroi,Resultado,Link do deck,Tipo de evento
0,Japan,21 Feb 2026,智彰 吉村 – Enigma – Heroes Scramble,ヒーローズスクランブル大阪,Silver Age,Enigma,1st,https://fabtcg.com/decklists/enigma/,outro
1,Japan,21 Feb 2026,"Masaaki Ochi – Ira, Crimson Haze – Heroes Scra...",ヒーローズスクランブル大阪,Silver Age,"Ira, Crimson Haze",2nd,https://fabtcg.com/decklists/masaaki-ochi-ira-...,outro
2,Japan,21 Feb 2026,Shoki Imacho – Oldhim – Heroes Scramble,ヒーローズスクランブル大阪,Silver Age,Oldhim,5th,https://fabtcg.com/decklists/shoki-imacho-oldhim/,outro
3,Philippines,20 Feb 2026,Justin Cu – Lyath Goldmane – Primed to Fight,Primed to Fight,Silver Age,Lyath Goldmane,nao informado,https://fabtcg.com/decklists/justin-cu-lyath-g...,primed to fight
4,Philippines,20 Feb 2026,Justin Cu – Zen – Primed to Fight,Primed to Fight,Silver Age,Zen,nao informado,https://fabtcg.com/decklists/justin-cu-zen-pri...,primed to fight
